Dependencies and Environment Variables

In [6]:
import os
import sys
sys.path.append("..")
import plotly.graph_objects as go
import plotly.express as px
from datasets import load_dataset

from src.model_loader import load_model
from src.layer_sweep import run_layer_sweep, plot_layer_accuracy_curve
from src.dashboard import generate_dashboard

Load Base Model and TOFU Dataset

In [7]:
model, tokenizer, device = load_model(
    "open-unlearning/tofu_Llama-3.2-1B-Instruct_full"
)

forget_data = load_dataset("locuslab/TOFU", "forget10")["train"]
retain_data = load_dataset("locuslab/TOFU", "retain90")["train"]

# Extract just the questions
forget_questions = [x["question"] for x in forget_data]
retain_questions = [x["question"] for x in retain_data]

print(f"Forget set: {len(forget_questions)} questions")
print(f"Retain set: {len(retain_questions)} questions")
print(f"\nExample forget question: {forget_questions[0]}")
print(f"Example retain question: {retain_questions[0]}")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/tofu_Llama-3.2-1B-Instruct_full
Device: mps | dtype: torch.float16
Params: 1.2B
Forget set: 400 questions
Retain set: 3600 questions

Example forget question: What is the full name of the author born in Taipei, Taiwan on 05/11/1991 who writes in the genre of leadership?
Example retain question: Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?


Run Sweep over Layers:
- Extract activations for each layer
- Extract performance metrics
- Create summary plots showing separability and projection onto first 2 PCA components

In [8]:
N_LAYERS = 16
SAVE_DIR = "../data/sweep_base"
metrics = run_layer_sweep(
    model=model,
    tokenizer=tokenizer,
    forget_questions=forget_questions,
    retain_questions=retain_questions,
    device=device,
    n_layers=N_LAYERS,
    save_dir=SAVE_DIR,
)

# Cell — display the accuracy curve inline
fig = plot_layer_accuracy_curve(metrics)
fig.show()


Layer sweep:   6%|▋         | 1/16 [00:26<06:41, 26.73s/it]

Saved probe to ../data/sweep_base/layer0/probe.joblib
layer  0 | train 1.000 | test 0.838 | gap 0.013 | overlap 0.463


Layer sweep:  12%|█▎        | 2/16 [00:53<06:18, 27.01s/it]

Saved probe to ../data/sweep_base/layer1/probe.joblib
layer  1 | train 1.000 | test 0.762 | gap 0.030 | overlap 0.875


Layer sweep:  19%|█▉        | 3/16 [01:20<05:49, 26.88s/it]

...

Saved probe to ../data/sweep_base/layer13/probe.joblib
layer 13 | train 1.000 | test 0.881 | gap 0.614 | overlap 0.672


Layer sweep:  94%|█████████▍| 15/16 [06:50<00:28, 28.19s/it]

Saved probe to ../data/sweep_base/layer14/probe.joblib
layer 14 | train 1.000 | test 0.969 | gap 1.224 | overlap 0.028


Layer sweep: 100%|██████████| 16/16 [07:17<00:00, 27.34s/it]

Saved probe to ../data/sweep_base/layer15/probe.joblib
layer 15 | train 1.000 | test 0.988 | gap 2.035 | overlap 0.003

Best layer by test accuracy: 15 (0.988)
Artifacts saved to ../data/sweep_base/


Generate the HTML dashboard

In [12]:
generate_dashboard(SAVE_DIR, out_filename="dashboard.html")

Dashboard written to ../data/sweep_base/dashboard.html


'../data/sweep_base/dashboard.html'

In [11]:
import importlib
import src.dashboard

importlib.reload(src.dashboard)
from src.dashboard import generate_dashboard # Re-import the specific function